# Generative day-to-night augmentation for PPE detection: complete experiment in one notebook

**Kaggle settings:** Accelerator = **GPU T4 x2**, Internet = **On**.

**Add inputs** (right panel, *Add Input*):
1. Dataset **`heonh0/daynight-cityview`** ("Unpaired Day and Night cityview images", 522 day / 227 night).
2. A dataset with the SFCHD test frames, built with `build_sfchd_testsets.py` (folders `sfchd_day/` and `sfchd_night/`).
3. *Only when resuming a stopped run:* the **output of the previous version of this notebook**.
   Finished parts (CycleGAN, translated images, completed YOLO runs) are detected and skipped.

**What it does:** (1) trains the day-to-night CycleGAN (`cyclegan_d2n.py`, configuration of `Gen_AI.py`); (2) translates the 1,132
Construction-PPE training images; (3) trains YOLOv8s for 3 arms x 3 seeds (Baseline, Enhanced, Size-matched control);
(4) tests every detector on the Construction-PPE test split, SFCHD day and SFCHD night; (5) writes all results to
`/kaggle/working/outputs.zip` and the trained models to `weights.zip`.

Run with **Save Version -> Save & Run All (Commit)** (12 h limit per run). CycleGAN (GPU 0) runs in parallel with the
Baseline and Size-matched YOLO runs (GPU 1); expected total about 3.5-5 h on T4 x2 with `CROP = 256` and 50 YOLO
epochs. `CROP = 512` reproduces the full resolution of `Gen_AI.py` but adds about 4 h.

In [ ]:
CROP = 256                  # 256 (fits one session) or 512 (Gen_AI.py resolution; needs a resumed second session)
CG_EPOCHS, CG_STEPS = 50, 200
SEEDS = [0, 1, 2]
ARMS = ['baseline', 'sizematch', 'enhanced']
EPOCHS, BATCH = 50, 32
PANEL_IMAGES = 'image801,image920,image22,image206'
!pip -q install ultralytics==8.4.82 || pip -q install -U ultralytics
import os
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'   # GPU numbers = nvidia-smi numbers in every process
import glob, json, time, shutil, subprocess, platform, torch
import ultralytics
WORK = '/kaggle/working'
os.chdir(WORK)
env = dict(python=platform.python_version(), torch=torch.__version__, cuda=torch.version.cuda,
           ultralytics=ultralytics.__version__,
           gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
json.dump(env, open('environment.json', 'w'), indent=1)
print(env)
# self-check: both GPUs must be visible and usable (Accelerator = GPU T4 x2)
for i in range(torch.cuda.device_count()):
    x = torch.ones(1024, 1024, device=f'cuda:{i}') @ torch.ones(1024, 1024, device=f'cuda:{i}')
    free, total = torch.cuda.mem_get_info(i)
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} OK, free {free / 1e9:.1f} of {total / 1e9:.1f} GB')
    del x
torch.cuda.empty_cache()
if torch.cuda.device_count() < 2:
    print('WARNING: only one GPU visible - set Accelerator to "GPU T4 x2" to run jobs in parallel')
print('CPU cores:', os.cpu_count())

In [ ]:
def find_dirs(name):
    return [h for h in glob.glob(f'/kaggle/input/**/{name}', recursive=True) if os.path.isdir(h)]

cands = [d for d in find_dirs('day') if os.path.isdir(os.path.join(os.path.dirname(d), 'night'))
         and len(os.listdir(d)) > 400]
assert cands, 'Add the dataset heonh0/daynight-cityview as an input'
DAYNIGHT = os.path.dirname(cands[0])
assert find_dirs('sfchd_night'), 'Add the SFCHD test-set dataset (sfchd_day/, sfchd_night/) as an input'
SFCHD = os.path.dirname(find_dirs('sfchd_night')[0])
print('day/night:', DAYNIGHT, '| SFCHD:', SFCHD)

# resume: bring finished parts from an attached previous output into /kaggle/working
for src in find_dirs('cyclegan_run'):
    if os.path.exists(f'{src}/DONE') and not os.path.exists(f'{WORK}/cyclegan_run/DONE'):
        shutil.copytree(src, f'{WORK}/cyclegan_run', dirs_exist_ok=True)
        print('resumed cyclegan_run from', src)
for src in find_dirs('ppe_night_train'):
    if len(os.listdir(src)) >= 1132 and not os.path.isdir(f'{WORK}/ppe_night_train'):
        shutil.copytree(src, f'{WORK}/ppe_night_train')
        print('resumed ppe_night_train from', src)
for src in find_dirs('panels'):
    if not os.path.isdir(f'{WORK}/panels'):
        shutil.copytree(src, f'{WORK}/panels')
for f in ('step1_timing.json', 'luminance_stats.json'):
    for src in glob.glob(f'/kaggle/input/**/{f}', recursive=True):
        if not os.path.exists(f'{WORK}/{f}'):
            shutil.copy(src, f'{WORK}/{f}')
for j in glob.glob('/kaggle/input/**/results/*_s[0-9].json', recursive=True):
    run = os.path.splitext(os.path.basename(j))[0]
    os.makedirs('results', exist_ok=True)
    if not os.path.exists(f'results/{run}.json'):
        shutil.copy(j, f'results/{run}.json')
        src_run = os.path.join(os.path.dirname(os.path.dirname(j)), 'yolo_runs', run)
        if os.path.isdir(src_run):
            shutil.copytree(src_run, f'yolo_runs/{run}', dirs_exist_ok=True)
        print('resumed', run)

In [ ]:
import urllib.request, zipfile
PPE = '/tmp/ppe'
if not glob.glob(f'{PPE}/**/images/train', recursive=True):
    os.makedirs(PPE, exist_ok=True)
    urllib.request.urlretrieve('https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip',
                               '/tmp/construction-ppe.zip')
    zipfile.ZipFile('/tmp/construction-ppe.zip').extractall(PPE)
PPE = os.path.dirname(os.path.dirname(glob.glob(f'{PPE}/**/images/train', recursive=True)[0]))
for s in ('train', 'val', 'test'):
    print(s, len(os.listdir(f'{PPE}/images/{s}')), 'images')

In [ ]:
%%writefile /kaggle/working/cyclegan_d2n.py
"""Day-to-night CycleGAN (PyTorch port of the repository script Gen_AI.py).

Configuration reproduced from Gen_AI.py:
- images resized (nearest neighbour) to 700 x 1200 (H x W), random 512 x 512 crop,
  random rot90 / flips for training, pixels scaled to [-1, 1];
- generator: 7x7 conv (64) -> three stride-2 3x3 convs (128, 256, 256) -> 6 residual
  blocks (conv-ReLU-conv, no normalisation) -> three 3x3 transposed convs with
  U-Net-style concatenation skips -> 7x7 conv + tanh; instance normalisation in the
  encoder/decoder blocks; weights ~ N(0, 0.02);
- discriminator: PatchGAN, 4x4 convs (64, 128, 256, 512 stride 2; 512 stride 1),
  instance normalisation, LeakyReLU(0.2), 4x4 output conv;
- losses: binary cross-entropy adversarial loss (discriminator loss halved) and L1
  cycle-consistency loss with lambda = 10;
- Adam, learning rate 2.5e-4, beta1 = 0.5, batch size 2, 200 steps per epoch,
  up to 50 epochs, early stopping (patience 10) that restores the best weights;
- seed 7.

Two behaviours of Gen_AI.py are reproduced as they actually executed:
- its LearningRateScheduler acted on the Keras model's default optimizer, not on
  the Adam optimizer used by train_step, so the effective learning rate was constant;
- its identity term compared an image with the output of the *other* generator and
  was differentiated only with respect to the generator not involved, so it
  contributed no gradient. It is therefore omitted here (identical updates).
Batch size 2 is realised directly (--bs 2) or as two accumulated single-image steps, which is exactly
equivalent because instance normalisation has no batch dependence and the losses are
batch means.
"""
import argparse
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

IMG = 512
RESIZE_HW = (700, 1200)  # overridden by --crop (resize scaled so crop/width ratio is preserved)
LAMBDA = 10.0
LR = 2.5e-4
SEED = 7


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.InstanceNorm2d) and m.affine:
        nn.init.normal_(m.weight, 0.0, 0.02)  # tfa gamma_initializer RandomNormal(0, 0.02)
        nn.init.zeros_(m.bias)


class Down(nn.Module):
    def __init__(self, cin, cout, k, s, act):
        super().__init__()
        if k == 4 and s == 1:  # TF 'same' padding for an even kernel: 1 before, 2 after
            self.pad = nn.ZeroPad2d((1, 2, 1, 2))
            p = 0
        else:
            self.pad = nn.Identity()
            p = (k - 1) // 2 if s == 1 else (k - s + 1) // 2
            if k == 3 and s == 2:
                p = 1
            if k == 4 and s == 2:
                p = 1
        self.conv = nn.Conv2d(cin, cout, k, s, p, bias=False)
        self.norm = nn.InstanceNorm2d(cout, affine=True)
        self.act = act

    def forward(self, x):
        return self.act(self.norm(self.conv(self.pad(x))))


class Up(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.conv = nn.ConvTranspose2d(cin, cout, 3, 2, 1, output_padding=1, bias=False)
        self.norm = nn.InstanceNorm2d(cout, affine=True)

    def forward(self, x):
        return F.relu(self.norm(self.conv(x)))


class Res(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c1 = nn.Conv2d(c, c, 3, 1, 1, bias=False)
        self.c2 = nn.Conv2d(c, c, 3, 1, 1, bias=False)

    def forward(self, x):
        return x + self.c2(F.relu(self.c1(x)))


class Generator(nn.Module):
    def __init__(self, n_res=6):
        super().__init__()
        r = nn.ReLU()
        self.e1 = Down(3, 64, 7, 1, r)
        self.e2 = Down(64, 128, 3, 2, r)
        self.e3 = Down(128, 256, 3, 2, r)
        self.e4 = Down(256, 256, 3, 2, r)
        self.res = nn.Sequential(*[Res(256) for _ in range(n_res)])
        self.u1 = Up(512, 256)
        self.u2 = Up(256 + 256, 128)
        self.u3 = Up(128 + 128, 64)
        self.out = nn.Conv2d(64 + 64, 3, 7, 1, 3, bias=False)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(e1)
        e3 = self.e3(e2)
        e4 = self.e4(e3)
        h = self.res(e4)
        h = self.u1(torch.cat([h, e4], 1))
        h = self.u2(torch.cat([h, e3], 1))
        h = self.u3(torch.cat([h, e2], 1))
        return torch.tanh(self.out(torch.cat([h, e1], 1)))


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        a = nn.LeakyReLU(0.2)
        self.body = nn.Sequential(
            Down(3, 64, 4, 2, a), Down(64, 128, 4, 2, a), Down(128, 256, 4, 2, a),
            Down(256, 512, 4, 2, a), Down(512, 512, 4, 1, a))
        self.out = nn.Conv2d(512, 1, 4, 1, 0)

    def forward(self, x):
        return self.out(self.body(x))


def _resize_one(path):
    im = Image.open(path).convert('RGB')
    return np.asarray(im.resize((RESIZE_HW[1], RESIZE_HW[0]), Image.NEAREST))


def preload(paths):
    """Decode and resize every image once (the Gen_AI.py resize step) and keep it in RAM."""
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as ex:
        return np.stack(list(ex.map(_resize_one, paths)))


def load_train(arr, rng):
    a = arr
    y = rng.integers(0, RESIZE_HW[0] - IMG + 1)
    x = rng.integers(0, RESIZE_HW[1] - IMG + 1)
    a = a[y:y + IMG, x:x + IMG]
    p = rng.random()
    if p > 0.8:
        a = np.rot90(a, 3)
    elif p > 0.6:
        a = np.rot90(a, 2)
    elif p > 0.4:
        a = np.rot90(a, 1)
    p = rng.random()
    if p > 0.7 and rng.random() < 0.5:
        a = a[:, ::-1]
    elif p < 0.3 and rng.random() < 0.5:
        a = a[::-1, :]
    return np.ascontiguousarray(a)  # uint8 HWC; scaled to [-1, 1] on the GPU


def to_gpu(batch, dev):
    t = torch.from_numpy(np.stack(batch)).pin_memory().to(dev, non_blocking=True)
    t = t.permute(0, 3, 1, 2).float().div_(127.5).sub_(1.0)
    return t


def bce(logits, target_one):
    t = torch.ones_like(logits) if target_one else torch.zeros_like(logits)
    return F.binary_cross_entropy_with_logits(logits, t)


def train(args):
    global IMG, RESIZE_HW
    IMG = args.crop
    RESIZE_HW = (round(700 * args.crop / 512), round(1200 * args.crop / 512))
    set_seed(SEED)
    rng = np.random.default_rng(SEED)
    dev = 'cuda'
    torch.backends.cudnn.benchmark = True
    day = preload(sorted(Path(args.data, 'day').glob('*.jpg')))
    night = preload(sorted(Path(args.data, 'night').glob('*.jpg')))
    day_paths = sorted(Path(args.data, 'day').glob('*.jpg'))
    print('preloaded', day.shape, night.shape, 'crop', IMG, flush=True)
    bs, accum = args.bs, 2 // args.bs
    G, Fn = Generator().to(dev), Generator().to(dev)          # G: day->night, Fn: night->day
    Dy, Dx = Discriminator().to(dev), Discriminator().to(dev)  # Dy judges night, Dx judges day
    for m in (G, Fn, Dy, Dx):
        m.apply(init_weights)
    optG = torch.optim.Adam(G.parameters(), LR, betas=(0.5, 0.999))
    optF = torch.optim.Adam(Fn.parameters(), LR, betas=(0.5, 0.999))
    optDy = torch.optim.Adam(Dy.parameters(), LR, betas=(0.5, 0.999))
    optDx = torch.optim.Adam(Dx.parameters(), LR, betas=(0.5, 0.999))
    scaler = torch.amp.GradScaler('cuda', enabled=False)  # fp32 training
    out = Path(args.out)
    (out / 'samples').mkdir(parents=True, exist_ok=True)
    log = []
    best = {'n2d': float('inf'), 'd2n': float('inf')}
    best_epoch, wait = 0, 0
    t0 = time.time()
    for epoch in range(1, args.epochs + 1):
        sums = {'gen_D2N_loss': 0.0, 'gen_N2D_loss': 0.0, 'disc_night_loss': 0.0,
                'disc_day_loss': 0.0, 'cycle_loss': 0.0}
        acc_t = torch.zeros(5, device=dev)
        for step in range(args.steps):
            for o in (optG, optF, optDy, optDx):
                o.zero_grad(set_to_none=True)
            for _ in range(accum):  # batch size 2 (directly, or as exact accumulation of 1+1)
                x = to_gpu([load_train(day[rng.integers(len(day))], rng) for _ in range(bs)], dev)
                y = to_gpu([load_train(night[rng.integers(len(night))], rng) for _ in range(bs)], dev)
                if True:
                    fake_y = G(x)
                    cyc_x = Fn(fake_y)
                    fake_x = Fn(y)
                    cyc_y = G(fake_x)
                    cyc = LAMBDA * ((x - cyc_x).abs().mean() + (y - cyc_y).abs().mean())
                    gG = bce(Dy(fake_y), True) + cyc
                    gF = bce(Dx(fake_x), True) + cyc
                # G's gradient comes from gG and F's from gF; each loss also depends on the
                # other generator through the shared cycle term, so compute them separately.
                g_params, f_params = list(G.parameters()), list(Fn.parameters())
                gradsG = torch.autograd.grad(scaler.scale(gG / accum), g_params, retain_graph=True)
                gradsF = torch.autograd.grad(scaler.scale(gF / accum), f_params)
                for p, g in zip(g_params, gradsG):
                    p.grad = g if p.grad is None else p.grad + g
                for p, g in zip(f_params, gradsF):
                    p.grad = g if p.grad is None else p.grad + g
                if True:
                    dY = 0.5 * (bce(Dy(y), True) + bce(Dy(fake_y.detach()), False))
                    dX = 0.5 * (bce(Dx(x), True) + bce(Dx(fake_x.detach()), False))
                scaler.scale((dY + dX) / accum).backward()
                acc_t += torch.stack([gG, gF, dY, dX, cyc]).detach().float() / accum
            for o in (optG, optF, optDy, optDx):
                scaler.step(o)
            scaler.update()
        vals = (acc_t / args.steps).tolist()
        row = dict(zip(['gen_D2N_loss', 'gen_N2D_loss', 'disc_night_loss', 'disc_day_loss', 'cycle_loss'], vals))
        row['epoch'] = epoch
        row['elapsed_min'] = (time.time() - t0) / 60
        log.append(row)
        print(json.dumps(row), flush=True)
        # Early stopping as in Gen_AI.py: both generator losses must improve.
        if row['gen_N2D_loss'] < best['n2d'] and row['gen_D2N_loss'] < best['d2n']:
            best = {'n2d': row['gen_N2D_loss'], 'd2n': row['gen_D2N_loss']}
            best_epoch, wait = epoch, 0
            torch.save({'G': G.state_dict(), 'F': Fn.state_dict(), 'epoch': epoch}, out / 'best.pt')
        else:
            wait += 1
        torch.save({'G': G.state_dict(), 'F': Fn.state_dict(), 'epoch': epoch}, out / 'last.pt')
        save_samples(G, Fn, day_paths[:4], out / 'samples' / f'epoch{epoch:03d}.jpg', dev)
        with open(out / 'cyclegan_log.json', 'w') as f:
            json.dump({'log': log, 'best_epoch': best_epoch, 'stopped_epoch': epoch,
                       'config': {'crop': IMG, 'resize_hw': RESIZE_HW, 'batch': 2, 'batch_per_pass': bs,
                                  'lr': LR, 'beta1': 0.5, 'lambda': LAMBDA, 'seed': SEED, 'max_epochs': args.epochs,
                                  'steps_per_epoch': args.steps, 'n_day': int(len(day)), 'n_night': int(len(night)),
                                  'precision': 'fp32', 'gpu': torch.cuda.get_device_name(0),
                                  'torch': torch.__version__}}, f, indent=1)
        if wait >= 10:
            print(f'early stopping at epoch {epoch}; best epoch {best_epoch}', flush=True)
            break
    (out / 'DONE').write_text(json.dumps({'best_epoch': best_epoch, 'stopped_epoch': epoch}))  # completion marker


@torch.no_grad()
def translate_array(G, a, dev, long_side=640):
    """Translate an HxWx3 uint8 array; returns the translated image at the resized size."""
    h, w = a.shape[:2]
    s = long_side / max(h, w)
    nh, nw = max(8, round(h * s)), max(8, round(w * s))
    im = np.asarray(Image.fromarray(a).resize((nw, nh), Image.BICUBIC), dtype=np.float32)
    ph, pw = (-nh) % 8, (-nw) % 8
    t = torch.from_numpy(im.transpose(2, 0, 1) / 127.5 - 1.0).unsqueeze(0).to(dev)
    if ph or pw:
        t = F.pad(t, (0, pw, 0, ph), mode='reflect')
    o = G(t)
    o = o[:, :, :nh, :nw].float().clamp(-1, 1)
    return ((o[0].cpu().numpy().transpose(1, 2, 0) + 1) * 127.5).round().astype(np.uint8)


def save_samples(G, Fn, paths, dst, dev):
    G.eval(); Fn.eval()
    rows = []
    for p in paths:
        a = np.asarray(Image.open(p).convert('RGB'))
        n = translate_array(G, a, dev, 384)
        c = translate_array(Fn, n, dev, 384)
        base = np.asarray(Image.fromarray(a).resize((n.shape[1], n.shape[0])))
        rows.append(np.concatenate([base, n, c], 1))
    wmax = max(r.shape[1] for r in rows)
    rows = [np.pad(r, ((0, 0), (0, wmax - r.shape[1]), (0, 0)), constant_values=255) for r in rows]
    Image.fromarray(np.concatenate(rows, 0)).save(dst, quality=90)
    G.train(); Fn.train()


def translate_dir(args):
    dev = 'cuda'
    ck = torch.load(args.weights, map_location=dev)
    G = Generator().to(dev)
    G.load_state_dict(ck['G'])
    G.eval()
    src, dst = Path(args.src), Path(args.dst)
    dst.mkdir(parents=True, exist_ok=True)
    files = sorted(p for p in src.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    for i, p in enumerate(files):
        a = np.asarray(Image.open(p).convert('RGB'))
        o = translate_array(G, a, dev, args.long_side)
        Image.fromarray(o).save(dst / (p.stem + '_n2.jpg'), quality=95)
        if i % 200 == 0:
            print(i, len(files), flush=True)
    print('translated', len(files), 'epoch', ck.get('epoch'))


@torch.no_grad()
def panels(args):
    """Qualitative panels: original | photometric darkening | CycleGAN G(x) | cycled F(G(x))."""
    dev = 'cuda'
    ck = torch.load(args.weights, map_location=dev)
    G, Fn = Generator().to(dev), Generator().to(dev)
    G.load_state_dict(ck['G']); Fn.load_state_dict(ck['F'])
    G.eval(); Fn.eval()
    dst = Path(args.dst); dst.mkdir(parents=True, exist_ok=True)
    rows = []
    for name in args.names.split(','):
        p = next(Path(args.src).glob(name + '.*'))
        a = np.asarray(Image.open(p).convert('RGB'))
        n = translate_array(G, a, dev, args.long_side)
        c = translate_array(Fn, n, dev, args.long_side)
        base = np.asarray(Image.fromarray(a).resize((n.shape[1], n.shape[0]), Image.BICUBIC))
        # photometric darkening equal to the strongest HSV value reduction of the detector's default augmentation (gain 0.6)
        import cv2
        hsv = cv2.cvtColor(base, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[..., 2] *= 0.6
        dark = cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2RGB)
        for tag, im in (('original', base), ('photometric', dark), ('cyclegan', n), ('cycled', c)):
            Image.fromarray(im).save(dst / f'{p.stem}_{tag}.png')
        rows.append(np.concatenate([base, dark, n, c], 1))
        diff = np.abs(base.astype(np.float32) - c.astype(np.float32)).mean()
        print(p.stem, 'mean |x - F(G(x))| =', round(float(diff), 2), flush=True)
    wmax = max(r.shape[1] for r in rows)
    rows = [np.pad(r, ((0, 6), (0, wmax - r.shape[1]), (0, 0)), constant_values=255) for r in rows]
    Image.fromarray(np.concatenate(rows, 0)).save(dst / 'panels_grid.jpg', quality=92)


def stats(args):
    """Mean luminance of image folders (0-255), for describing the translated images."""
    out = {}
    for d in args.dirs.split(','):
        vals = []
        for p in sorted(Path(d).iterdir()):
            if p.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
                continue
            im = Image.open(p).convert('L'); im.thumbnail((320, 320))
            vals.append(float(np.asarray(im, dtype=np.float32).mean()))
        v = np.array(vals)
        out[d] = dict(n=len(v), median=float(np.median(v)), q1=float(np.percentile(v, 25)), q3=float(np.percentile(v, 75)))
        print(d, out[d], flush=True)
    Path(args.json).write_text(json.dumps(out, indent=1))


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest='cmd', required=True)
    t = sub.add_parser('train')
    t.add_argument('--data', required=True)
    t.add_argument('--out', required=True)
    t.add_argument('--epochs', type=int, default=50)
    t.add_argument('--steps', type=int, default=200)
    t.add_argument('--bs', type=int, default=1, choices=[1, 2])
    t.add_argument('--crop', type=int, default=512)
    r = sub.add_parser('translate')
    r.add_argument('--weights', required=True)
    r.add_argument('--src', required=True)
    r.add_argument('--dst', required=True)
    r.add_argument('--long_side', type=int, default=640)
    q = sub.add_parser('panels')
    q.add_argument('--weights', required=True)
    q.add_argument('--src', required=True)
    q.add_argument('--dst', required=True)
    q.add_argument('--names', required=True, help='comma-separated image stems')
    q.add_argument('--long_side', type=int, default=640)
    st = sub.add_parser('stats')
    st.add_argument('--dirs', required=True)
    st.add_argument('--json', required=True)
    a = ap.parse_args()
    {'train': train, 'translate': translate_dir, 'panels': panels, 'stats': stats}[a.cmd](a)

In [ ]:
%%writefile /kaggle/working/run_yolo.py
"""Controlled YOLOv8s experiment: Baseline vs Enhanced (CycleGAN night) vs Size-matched control.

Arms (same Construction-PPE val split for checkpoint selection, same hyperparameters):
  baseline  : Construction-PPE training images (n = 1,132)
  enhanced  : training images + their CycleGAN day-to-night translations (n = 2,264)
  sizematch : training images listed twice (n = 2,264), to control for training-set size
Test sets (never used for training or checkpoint selection):
  ppe_test    : Construction-PPE test split (141 images, 11 classes)
  sfchd_day   : 447 daytime frames, SFCHD outdoor cameras (Person, helmet)
  sfchd_night : 447 night-time frames, same cameras (Person, helmet)
Outputs under --out: yolo_runs/<arm>_s<seed>/ (results.csv, weights, val outputs) and results/<arm>_s<seed>.json
"""
import argparse
import json
import os
import platform
import shutil
import time
from pathlib import Path

NAMES = ['helmet', 'gloves', 'vest', 'boots', 'goggles', 'none', 'Person',
         'no_helmet', 'no_goggle', 'no_gloves', 'no_boots']
HYP = dict(epochs=50, imgsz=640, optimizer='AdamW', lr0=0.001, weight_decay=0.0005,
           cos_lr=True, warmup_epochs=3, warmup_momentum=0.8, deterministic=False)


def link(src, dst):
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def prepare(data, synth, work, arm):
    """Build the training list for one arm (each image a distinct file with its own label file)."""
    data, work = Path(data), Path(work)
    work.mkdir(parents=True, exist_ok=True)
    imgs = sorted(p for p in (data / 'images/train').iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    base = [str(p) for p in imgs]
    extra = []
    if arm in ('enhanced', 'sizematch'):
        tag = 'night' if arm == 'enhanced' else 'dup'
        (work / f'{tag}/images').mkdir(parents=True, exist_ok=True)
        (work / f'{tag}/labels').mkdir(parents=True, exist_ok=True)
        missing = []
        for p in imgs:
            lab = data / 'labels/train' / (p.stem + '.txt')
            if tag == 'night':
                s = Path(synth) / (p.stem + '_n2.jpg')
                if not s.exists():
                    missing.append(s.name)
                    continue
                link(s, work / f'night/images/{p.stem}_n2.jpg')
                link(lab, work / f'night/labels/{p.stem}_n2.txt')
                extra.append(str(work / f'night/images/{p.stem}_n2.jpg'))
            else:
                link(p, work / f'dup/images/{p.stem}_dup{p.suffix}')
                link(lab, work / f'dup/labels/{p.stem}_dup.txt')
                extra.append(str(work / f'dup/images/{p.stem}_dup{p.suffix}'))
        if missing:
            raise FileNotFoundError(f'{len(missing)} translated images missing, e.g. {missing[:3]}')
    L = base + extra
    (work / f'{arm}.txt').write_text(chr(10).join(L) + chr(10))
    yaml = {'path': str(work), 'train': str(work / f'{arm}.txt'),
            'val': str(data / 'images/val'), 'names': dict(enumerate(NAMES))}
    (work / f'{arm}.yaml').write_text(json.dumps(yaml))  # JSON is valid YAML
    print(arm, len(L), 'training images', flush=True)
    return work / f'{arm}.yaml'


def test_yamls(data, sfchd_root, work, tag, extra=()):
    data, sfchd_root, work = Path(data), Path(sfchd_root), Path(work)
    out = {}
    sets = [('ppe_test', data / 'images/test'),
            ('sfchd_day', sfchd_root / 'sfchd_day/images'),
            ('sfchd_night', sfchd_root / 'sfchd_night/images')]
    for e in extra:   # additional test sets 'name=folder' (folder has images/ and labels/ in Construction-PPE ids)
        n, _, p = e.partition('=')
        sets.append((n, Path(p) / 'images'))
    for name, img in sets:
        if not img.exists():
            raise FileNotFoundError(img)
        y = work / f'test_{name}_{tag}.yaml'
        y.write_text(json.dumps({'path': str(img.parent), 'train': str(img), 'val': str(img),
                                 'names': dict(enumerate(NAMES))}))
        out[name] = y
    return out


def metrics_dict(m):
    b = m.box
    per = {}
    for i, c in enumerate(b.ap_class_index):
        per[NAMES[int(c)]] = dict(P=float(b.p[i]), R=float(b.r[i]), F1=float(b.f1[i]),
                                  mAP50=float(b.ap50[i]), mAP50_95=float(b.ap[i]))
    P, R = float(b.mp), float(b.mr)
    return dict(P=P, R=R, F1=(2 * P * R / (P + R) if P + R else 0.0), mAP50=float(b.map50),
                mAP50_95=float(b.map), per_class=per, speed=m.speed)


def env_info():
    import torch
    import ultralytics
    return dict(python=platform.python_version(), torch=torch.__version__, ultralytics=ultralytics.__version__,
                cuda=torch.version.cuda, gpu=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data', required=True, help='Construction-PPE root (images/, labels/)')
    ap.add_argument('--synth', default='', help='folder of CycleGAN night translations (*_n2.jpg)')
    ap.add_argument('--sfchd', required=True, help='folder containing sfchd_day/ and sfchd_night/')
    ap.add_argument('--out', default='.')
    ap.add_argument('--work', default=None)
    ap.add_argument('--arms', default='baseline,enhanced,sizematch')
    ap.add_argument('--seeds', default='0,1,2')
    ap.add_argument('--batch', type=int, default=16)
    ap.add_argument('--workers', type=int, default=4)
    ap.add_argument('--amp', type=int, default=1)
    ap.add_argument('--epochs', type=int, default=HYP['epochs'])
    ap.add_argument('--fraction', type=float, default=1.0)
    ap.add_argument('--tag', default='')
    ap.add_argument('--device', default='0')
    ap.add_argument('--cache', default='ram')
    ap.add_argument('--prepare_only', action='store_true')
    ap.add_argument('--extra', action='append', default=[], help='extra test set name=folder')
    a = ap.parse_args()
    out = Path(a.out).resolve()
    work = Path(a.work) if a.work else out / 'yolo_data'
    if a.prepare_only:
        for arm in a.arms.split(','):
            prepare(a.data, a.synth, work, arm)
        return
    from ultralytics import YOLO
    res_dir = out / 'results'
    res_dir.mkdir(parents=True, exist_ok=True)
    for seed in [int(s) for s in a.seeds.split(',')]:
        for arm in a.arms.split(','):
            name = f'{arm}_s{seed}{a.tag}'
            out_json = res_dir / f'{name}.json'
            if out_json.exists():
                print('skip', name, flush=True)
                continue
            t0 = time.time()
            data_yaml = prepare(a.data, a.synth, work, arm)
            tests = test_yamls(a.data, a.sfchd, work, name, a.extra)
            model = YOLO('yolov8s.pt')
            hyp = dict(HYP, epochs=a.epochs)
            model.train(data=str(data_yaml), batch=a.batch, workers=a.workers,
                        seed=seed, amp=bool(a.amp), project=str(out / 'yolo_runs'), name=name, device=a.device,
                        cache=(a.cache if a.cache != 'none' else False),
                        exist_ok=True, fraction=a.fraction, plots=False, verbose=False, **hyp)
            train_min = (time.time() - t0) / 60
            best = out / 'yolo_runs' / name / 'weights' / 'best.pt'
            ev = {}
            m = YOLO(str(best))
            for tname, y in tests.items():
                r = m.val(data=str(y), split='val', imgsz=640, batch=a.batch, conf=0.001, iou=0.7,
                          device=a.device, plots=False, save_json=True, verbose=False,
                          project=str(out / 'yolo_runs' / name), name=f'val_{tname}', exist_ok=True)
                ev[tname] = metrics_dict(r)
            ev.update(train_minutes=train_min, total_minutes=(time.time() - t0) / 60, arm=arm, seed=seed,
                      batch=a.batch, amp=bool(a.amp), epochs=a.epochs, env=env_info())
            out_json.write_text(json.dumps(ev, indent=1))
            print(name, json.dumps({k: {kk: round(vv, 4) for kk, vv in v.items() if kk in ('mAP50', 'mAP50_95', 'P', 'R')}
                                    for k, v in ev.items() if isinstance(v, dict) and 'mAP50' in v}), flush=True)


if __name__ == '__main__':
    main()

## Run everything
CycleGAN training and translation (GPU 0) run **in parallel** with the Baseline and Size-matched YOLO runs, which do
not need translated images (GPU 1). Enhanced runs start as soon as the 1,132 translated images exist; then both GPUs
share the remaining runs. Progress is printed every few minutes; detailed logs are in `logs/`.

In [ ]:
os.makedirs('logs', exist_ok=True)
ngpu = max(1, torch.cuda.device_count())
workers = 2 if ngpu > 1 else 4
from ultralytics import YOLO
YOLO('yolov8s.pt')   # download COCO-pretrained weights once before parallel jobs start
T0 = time.time()


def now():
    return time.strftime('%H:%M') + f' (+{(time.time() - T0) / 3600:.2f} h)'


def n_translated():
    return len(os.listdir('ppe_night_train')) if os.path.isdir('ppe_night_train') else 0


def tail(path, n=25):
    try:
        return ''.join(open(path, errors='ignore').readlines()[-n:])
    except FileNotFoundError:
        return '(no log yet)'


def cg_state():
    """Human-readable state of the CycleGAN job (training -> translation -> panels -> statistics)."""
    if cg_proc is None:
        return 'CycleGAN: already complete (reused)'
    if cg_proc.poll() is not None:
        return 'CycleGAN: FINISHED' if cg_proc.returncode == 0 else f'CycleGAN: FAILED (exit {cg_proc.returncode})'
    if os.path.exists('logs/t_translate_end'):
        return 'CycleGAN step 3/3: saving example panels and luminance statistics'
    if os.path.exists('logs/t_translate_start'):
        return f'CycleGAN step 2/3: translating PPE training images {n_translated()}/1132'
    ep_lines = [l for l in tail('logs/cyclegan.log', 400).splitlines() if l.startswith('{"gen_D2N_loss"')]
    if ep_lines:
        r = json.loads(ep_lines[-1])
        rate = r['elapsed_min'] / r['epoch']
        return (f"CycleGAN step 1/3: training epoch {r['epoch']}/{CG_EPOCHS} (cycle loss {r['cycle_loss']:.2f}; "
                f"{rate:.1f} min/epoch; at most {rate * (CG_EPOCHS - r['epoch']):.0f} min left, early stopping may end sooner)")
    return 'CycleGAN step 1/3: loading images / first epoch running'


def yolo_epoch(arm, seed):
    f = f'yolo_runs/{arm}_s{seed}/results.csv'
    return max(0, sum(1 for _ in open(f)) - 1) if os.path.exists(f) else 0


def status():
    done = len(glob.glob('results/*_s[0-9].json'))
    lines = [f'==== {now()} | YOLO runs finished {done}/{len(SEEDS) * len(ARMS)} | queued {len(jobs)} ====',
             '  ' + cg_state()]
    for g in range(ngpu):
        if g in running:
            p, arm, seed, t0 = running[g]
            ep = yolo_epoch(arm, seed)
            eta = ((time.time() - t0) / 60 / ep * (EPOCHS - ep)) if ep else None
            lines.append(f'  GPU {g}: YOLO {arm} seed {seed} - epoch {ep}/{EPOCHS}'
                         + (f', about {eta:.0f} min left for this run' if eta is not None else ' (caching images / starting)'))
        elif g == 0 and cg_proc is not None and cg_proc.poll() is None:
            lines.append('  GPU 0: CycleGAN job')
        else:
            lines.append(f'  GPU {g}: idle' + (' (Enhanced runs wait for the translated images)'
                                               if jobs and all(a == 'enhanced' for a, _ in jobs) else ''))
    try:
        smi = subprocess.run(['nvidia-smi', '--query-gpu=index,utilization.gpu,memory.used',
                              '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=20).stdout
        lines.append('  nvidia-smi: ' + ' | '.join(f'GPU {a.strip()}: {b.strip()}% util, {c.strip()} MiB'
                                                   for a, b, c in (l.split(',') for l in smi.strip().splitlines())))
        apps = subprocess.run(['nvidia-smi', '--query-compute-apps=gpu_bus_id,pid,used_memory',
                               '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=20).stdout
        bus = subprocess.run(['nvidia-smi', '--query-gpu=index,pci.bus_id', '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=20).stdout
        bus_idx = {b.strip(): i.strip() for i, b in (l.split(',') for l in bus.strip().splitlines())}
        for a in apps.strip().splitlines():
            b, pid, mem = [x.strip() for x in a.split(',')]
            try:
                cmdline = open(f'/proc/{pid}/cmdline').read().replace('\0', ' ')
            except OSError:
                cmdline = '?'
            job = ('CycleGAN' if 'cyclegan_d2n' in cmdline else
                   ' '.join(cmdline.split('--arms')[1].split()[:1] + ['seed'] + cmdline.split('--seeds')[1].split()[:1])
                   if '--arms' in cmdline else 'notebook kernel')
            lines.append(f'  on GPU {bus_idx.get(b, b)}: {job} ({mem} MiB)')
    except Exception as e:
        lines.append(f'  (nvidia-smi not available: {e})')
    print('\n'.join(lines), flush=True)


# ---- STEP 1: CycleGAN (training, translation, panels, statistics) as a background job on GPU 0 ----
cg_proc = None
if n_translated() < 1132 or not os.path.exists('luminance_stats.json'):
    for f in ('logs/t_translate_start', 'logs/t_translate_end'):   # remove markers left by an interrupted session
        if os.path.exists(f):
            os.remove(f)
    steps = []
    if not os.path.exists('cyclegan_run/DONE'):   # train (again) unless a completed run exists
        steps.append(f'python cyclegan_d2n.py train --data "{DAYNIGHT}" --out cyclegan_run --epochs {CG_EPOCHS} '
                     f'--steps {CG_STEPS} --bs 2 --crop {CROP}')
    if n_translated() < 1132:
        steps += ['date +%s > logs/t_translate_start',
                  f'python cyclegan_d2n.py translate --weights cyclegan_run/best.pt --src "{PPE}/images/train" '
                  f'--dst ppe_night_train --long_side 640',
                  'date +%s > logs/t_translate_end']
    steps += [f'python cyclegan_d2n.py panels --weights cyclegan_run/best.pt --src "{PPE}/images/train" --dst panels '
              f'--names "{PANEL_IMAGES}"',
              f'python cyclegan_d2n.py stats --dirs "{PPE}/images/train,{WORK}/ppe_night_train,{SFCHD}/sfchd_day/images,'
              f'{SFCHD}/sfchd_night/images" --json luminance_stats.json']
    cg_proc = subprocess.Popen(['bash', '-c', ' && '.join(steps)], env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'),
                               stdout=open('logs/cyclegan.log', 'w'), stderr=subprocess.STDOUT, cwd=WORK)
    print(now(), 'STEP 1 started: CycleGAN job on GPU 0 (log: logs/cyclegan.log)', flush=True)
else:
    print(now(), 'STEP 1 skipped: CycleGAN outputs already present', flush=True)

# ---- STEP 2: YOLO runs; Baseline and Size-matched start at once, Enhanced after translation ----
order = {'sizematch': 0, 'baseline': 1, 'enhanced': 2}
jobs = sorted([(a, s) for s in SEEDS for a in ARMS if not os.path.exists(f'results/{a}_s{s}.json')],
              key=lambda j: (order[j[0]], j[1]))
running, last_print, cg_logged, retries = {}, 0, cg_proc is None, {}
print(now(), f'STEP 2 started: {len(jobs)} YOLO runs on {ngpu} GPU(s); status every 2 min', flush=True)
while jobs or running or (cg_proc is not None and cg_proc.poll() is None):
    cg_busy = cg_proc is not None and cg_proc.poll() is None
    if cg_proc is not None and not cg_busy and not cg_logged:
        cg_logged = True
        if cg_proc.returncode != 0:
            print(now(), 'CycleGAN job FAILED - last lines of logs/cyclegan.log:\n' + tail('logs/cyclegan.log'), flush=True)
            if n_translated() < 1132:
                dropped = [j for j in jobs if j[0] == 'enhanced']
                jobs = [j for j in jobs if j[0] != 'enhanced']
                print(now(), f'Enhanced runs cannot start without translated images; dropped {dropped}', flush=True)
        else:
            timing = {'n_translated': n_translated(), 'crop': CROP, 'job_hours': (time.time() - T0) / 3600}
            if os.path.exists('cyclegan_run/cyclegan_log.json'):
                log = json.load(open('cyclegan_run/cyclegan_log.json'))
                timing.update(train_hours=log['log'][-1]['elapsed_min'] / 60, stopped_epoch=log['stopped_epoch'],
                              best_epoch=log['best_epoch'])
            if os.path.exists('logs/t_translate_end'):
                t_tr = [int(open(f'logs/{f}').read()) for f in ('t_translate_start', 't_translate_end')]
                timing['translate_minutes'] = (t_tr[1] - t_tr[0]) / 60
            json.dump(timing, open('step1_timing.json', 'w'), indent=1)
            print(now(), 'STEP 1 finished: CycleGAN job complete', timing, flush=True)
    translated = (not cg_busy) and n_translated() >= 1132
    for g in range(ngpu):
        if g in running or (g == 0 and cg_busy):
            continue
        for idx, (arm, seed) in enumerate(jobs):
            if arm == 'enhanced' and not translated:
                continue
            jobs.pop(idx)
            cmd = ['python', 'run_yolo.py', '--data', PPE, '--synth', f'{WORK}/ppe_night_train', '--sfchd', SFCHD,
                   '--out', WORK, '--arms', arm, '--seeds', str(seed), '--epochs', str(EPOCHS), '--batch', str(BATCH),
                   '--workers', str(workers), '--device', str(g), '--amp', '1']
            # Ultralytics selects the GPU itself from --device (it sets CUDA_VISIBLE_DEVICES internally), so the
            # physical GPU number is passed and no CUDA_VISIBLE_DEVICES is inherited from the notebook.
            yenv = {k: v for k, v in os.environ.items() if k != 'CUDA_VISIBLE_DEVICES'}
            running[g] = (subprocess.Popen(cmd, env=yenv,
                                           stdout=open(f'logs/{arm}_s{seed}.log', 'w'), stderr=subprocess.STDOUT, cwd=WORK),
                          arm, seed, time.time())
            print(now(), f'start YOLO {arm} seed {seed} on GPU {g} (log: logs/{arm}_s{seed}.log)', flush=True)
            time.sleep(20)   # stagger starts (dataset caching and AMP-check downloads)
            break
    for g, (p, arm, seed, t0) in list(running.items()):
        if p.poll() is not None:
            ok = p.returncode == 0 and os.path.exists(f'results/{arm}_s{seed}.json')
            print(now(), f'{"done" if ok else "FAILED"} YOLO {arm} seed {seed} (exit {p.returncode}) in '
                  f'{(time.time() - t0) / 60:.1f} min', flush=True)
            del running[g]
            if ok:
                r = json.load(open(f'results/{arm}_s{seed}.json'))
                print('   mAP@0.5  PPE test %.3f | SFCHD day %.3f | SFCHD night %.3f' %
                      (r['ppe_test']['mAP50'], r['sfchd_day']['mAP50'], r['sfchd_night']['mAP50']), flush=True)
            else:
                print('--- last lines of', f'logs/{arm}_s{seed}.log', '---\n' + tail(f'logs/{arm}_s{seed}.log'), flush=True)
                retries[(arm, seed)] = retries.get((arm, seed), 0) + 1
                if retries[(arm, seed)] <= 1:
                    jobs.insert(0, (arm, seed))   # retry once
                    print(now(), 'retrying', arm, seed, flush=True)
            last_print = 0   # print a status block right after every event
    if time.time() - last_print > 120:
        last_print = time.time()
        status()
    time.sleep(15)
status()
print(now(), 'STEP 2 finished: all YOLO runs done', flush=True)
from IPython.display import Image as IPImage, display
if os.path.exists('panels/panels_grid.jpg'):
    display(IPImage('panels/panels_grid.jpg', width=1100))

In [ ]:
print(time.strftime("%H:%M"), "STEP 3 started: summary table and output zips", flush=True)
import pandas as pd
rows = []
for j in sorted(glob.glob('results/*_s[0-9].json')):
    r = json.load(open(j))
    for t in ('ppe_test', 'sfchd_day', 'sfchd_night'):
        rows.append(dict(arm=r['arm'], seed=r['seed'], test=t,
                         **{k: r[t][k] for k in ('P', 'R', 'F1', 'mAP50', 'mAP50_95')}, train_min=r['train_minutes']))
df = pd.DataFrame(rows)
df.to_csv('summary.csv', index=False)
print(df.groupby(['test', 'arm'])[['P', 'R', 'F1', 'mAP50', 'mAP50_95']].agg(['mean', 'std']).round(3))
shutil.rmtree('yolo_data', ignore_errors=True)   # large intermediate copies (rebuilt from inputs when needed)
# one zip with all result files (no weights, no image copies)
with zipfile.ZipFile('outputs.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for pat in ['environment.json', 'step1_timing.json', 'luminance_stats.json', 'summary.csv', 'results/*.json',
                'logs/*.log', 'cyclegan_run/cyclegan_log.json', 'cyclegan_run/samples/*', 'panels/*',
                'yolo_runs/*/results.csv', 'yolo_runs/*/args.yaml', 'yolo_runs/*/val_*/predictions.json',
                'yolo_runs/*/val_*/*.png', 'cyclegan_d2n.py', 'run_yolo.py']:
        for f in glob.glob(pat):
            z.write(f)
print('outputs.zip', round(os.path.getsize('outputs.zip') / 1e6, 1), 'MB')
# trained weights (for any later evaluation without retraining)
with zipfile.ZipFile('weights.zip', 'w', zipfile.ZIP_STORED) as z:
    for f in glob.glob('yolo_runs/*/weights/best.pt') + ['cyclegan_run/best.pt']:
        if os.path.exists(f):
            z.write(f)
print('weights.zip', round(os.path.getsize('weights.zip') / 1e6, 1), 'MB')
print(time.strftime("%H:%M"), "STEP 3 finished: download outputs.zip and weights.zip from the Output tab", flush=True)